# London House Price Prediction
## Prompt A — Run 1

This notebook builds a machine learning model to predict London property sale prices
using property-level and area-level features.

**Stages:**
1. Data Ingestion and Quality Assessment
2. Exploratory Data Analysis
3. Baseline Model
4. Performance Improvement


In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print("All libraries loaded successfully.")


All libraries loaded successfully.


---
# STAGE 1 — Data Ingestion and Quality Assessment


In [2]:
# Load both datasets
prices = pd.read_csv('london_house_prices.csv')
areas = pd.read_csv('london_area_features.csv')

print("=" * 60)
print("HOUSE PRICES DATASET")
print("=" * 60)
print(f"Shape: {prices.shape}")
print(f"\nColumns: {list(prices.columns)}")
print(f"\nData types:")
print(prices.dtypes)
print(f"\nFirst 5 rows:")
prices.head()


HOUSE PRICES DATASET
Shape: (417561, 12)

Columns: ['outcode', 'latitude', 'longitude', 'bedrooms', 'bathrooms', 'floorAreaSqM', 'livingRooms', 'propertyType', 'tenure', 'energyRating', 'rentEstimate', 'price']

Data types:
outcode          object
latitude        float64
longitude       float64
bedrooms        float64
bathrooms       float64
floorAreaSqM    float64
livingRooms     float64
propertyType     object
tenure           object
energyRating     object
rentEstimate    float64
price           float64
dtype: object

First 5 rows:


,outcode,latitude,longitude,bedrooms,bathrooms,floorAreaSqM,livingRooms,propertyType,tenure,energyRating,rentEstimate,price
0,EC4A,51.517282,-0.110314,1.0,1.0,45.0,1.0,Purpose Built Flat,Leasehold,NaN,2350.0,600000.0
1,EC4A,51.517282,-0.110314,NaN,NaN,NaN,NaN,Flat/Maisonette,Leasehold,NaN,2350.0,600000.0
2,SW1P,51.495505,-0.132379,2.0,2.0,71.0,1.0,Flat/Maisonette,Leasehold,C,2950.0,759000.0
3,SE5,51.478185,-0.092201,1.0,1.0,64.0,1.0,Flat/Maisonette,Leasehold,D,2000.0,388000.0
4,N10,51.588774,-0.139599,4.0,1.0,137.0,2.0,End Terrace House,Freehold,D,4850.0,1261000.0


In [3]:
print("=" * 60)
print("AREA FEATURES DATASET")
print("=" * 60)
print(f"Shape: {areas.shape}")
print(f"\nColumns ({len(areas.columns)} total):")
for i, col in enumerate(areas.columns):
    print(f"  {i+1}. {col}")
print(f"\nData types:")
print(areas.dtypes)


AREA FEATURES DATASET
Shape: (168, 52)

Columns (52 total):
  1. outcode
  2. outcode_lat
  3. outcode_lon
  4. n_properties
  5. crime_anti_social_behaviour
  6. crime_bicycle_theft
  7. crime_burglary
  8. crime_criminal_damage_and_arson
  9. crime_drugs
  10. crime_other_crime
  11. crime_other_theft
  12. crime_possession_of_weapons
  13. crime_public_order
  14. crime_robbery
  15. crime_shoplifting
  16. crime_theft_from_the_person
  17. crime_vehicle_crime
  18. crime_violence_and_sexual_offences
  19. crime_total
  20. census_denom_total
  21. census_employed_total_perc
  22. census_retired_perc
  23. census_unemployed_perc
  24. census_age_16_to_34_perc
  25. census_age_65_plus_perc
  26. census_level4_perc
  27. census_no_qualifications_perc
  28. poi_bakery
  29. poi_bank
  30. poi_bar
  31. poi_bus_station
  32. poi_cafe
  33. poi_clinic
  34. poi_community_centre
  35. poi_conference_centre
  36. poi_coworking_space
  37. poi_disused
  38. poi_doctors
  39. poi_dojo
  40. 

In [4]:
print("=" * 60)
print("MISSING VALUES ANALYSIS")
print("=" * 60)

print("\n--- House Prices Dataset ---")
missing_prices = prices.isnull().sum()
missing_pct = (missing_prices / len(prices) * 100).round(2)
missing_report = pd.DataFrame({
    'Missing Count': missing_prices,
    'Missing %': missing_pct
})
print(missing_report[missing_report['Missing Count'] > 0].sort_values('Missing %', ascending=False))

print("\n--- Area Features Dataset ---")
missing_areas = areas.isnull().sum()
if missing_areas.sum() > 0:
    missing_pct_areas = (missing_areas / len(areas) * 100).round(2)
    missing_report_areas = pd.DataFrame({
        'Missing Count': missing_areas,
        'Missing %': missing_pct_areas
    })
    print(missing_report_areas[missing_report_areas['Missing Count'] > 0])
else:
    print("No missing values in area features dataset.")


MISSING VALUES ANALYSIS

--- House Prices Dataset ---
              Missing Count  Missing %
energyRating          84288      20.19
bathrooms             77755      18.62
livingRooms           60341      14.45
bedrooms              40404       9.68
floorAreaSqM          25066       6.00
tenure                11494       2.75
propertyType           1126       0.27
rentEstimate           1101       0.26

--- Area Features Dataset ---
No missing values in area features dataset.


In [5]:
print("=" * 60)
print("PRICE OUTLIER ANALYSIS")
print("=" * 60)

print("\nPrice statistics:")
print(prices['price'].describe().apply(lambda x: f"\u00a3{x:,.0f}" if abs(x) > 1 else f"{x:.4f}"))

print(f"\nSkewness: {prices['price'].skew():.2f}")
print(f"Kurtosis: {prices['price'].kurtosis():.2f}")

# IQR method
Q1 = prices['price'].quantile(0.25)
Q3 = prices['price'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers_iqr = prices[(prices['price'] < lower_bound) | (prices['price'] > upper_bound)]

print(f"\nIQR Outlier Detection:")
print(f"  Q1 = \u00a3{Q1:,.0f}, Q3 = \u00a3{Q3:,.0f}, IQR = \u00a3{IQR:,.0f}")
print(f"  Lower bound: \u00a3{max(0, lower_bound):,.0f}")
print(f"  Upper bound: \u00a3{upper_bound:,.0f}")
print(f"  Outliers: {len(outliers_iqr):,} ({len(outliers_iqr)/len(prices)*100:.1f}%)")

# Percentile-based
p99 = prices['price'].quantile(0.99)
p01 = prices['price'].quantile(0.01)
print(f"\n  1st percentile: \u00a3{p01:,.0f}")
print(f"  99th percentile: \u00a3{p99:,.0f}")

print(f"\nThe price distribution is HIGHLY RIGHT-SKEWED (skewness={prices['price'].skew():.1f}).")
print("Extreme outliers exist at the upper end (multi-million pound properties).")
print("Strategy: Cap outliers at 99th percentile and apply log-transform.")


PRICE OUTLIER ANALYSIS

Price statistics:
count       £417,561
mean        £904,519
std         £920,292
min          £89,000
25%         £446,000
50%         £622,000
75%         £984,000
max      £29,220,000
Name: price, dtype: object

Skewness: 5.05


Kurtosis: 50.65



IQR Outlier Detection:


  Q1 = £446,000, Q3 = £984,000, IQR = £538,000
  Lower bound: £0
  Upper bound: £1,791,000
  Outliers: 37,210 (8.9%)

  1st percentile: £228,000
  99th percentile: £4,959,000

The price distribution is HIGHLY RIGHT-SKEWED (skewness=5.1).
Extreme outliers exist at the upper end (multi-million pound properties).
Strategy: Cap outliers at 99th percentile and apply log-transform.


In [6]:
print("=" * 60)
print("MERGING DATASETS")
print("=" * 60)

df = prices.merge(areas, on='outcode', how='left')
print(f"Merged dataset shape: {df.shape}")
print(f"Rows before merge: {len(prices):,}")
print(f"Rows after merge:  {len(df):,}")
print(f"Rows lost: {len(prices) - len(df):,}")

# Check unmatched outcodes
area_cols = [c for c in areas.columns if c != 'outcode']
unmatched_mask = df[area_cols[0]].isnull()
unmatched_count = unmatched_mask.sum()
print(f"\nRows without area data: {unmatched_count:,} ({unmatched_count/len(df)*100:.1f}%)")
if unmatched_count > 0:
    print(f"Unmatched outcodes: {df.loc[unmatched_mask, 'outcode'].nunique()}")

print(f"\nFinal dataset info:")
print(f"  Total rows: {len(df):,}")
print(f"  Total columns: {len(df.columns)}")


MERGING DATASETS


Merged dataset shape: (417561, 63)
Rows before merge: 417,561
Rows after merge:  417,561
Rows lost: 0

Rows without area data: 0 (0.0%)

Final dataset info:
  Total rows: 417,561
  Total columns: 63


### Data Quality Issues and Target Leakage Detection


In [7]:
print("=" * 60)
print("DATA QUALITY ASSESSMENT")
print("=" * 60)

print("\n1. MISSING VALUES:")
print("   - Several property features have missing values (bedrooms, bathrooms, etc.)")
print("   - energyRating has high missingness")
print("   - Strategy: Median imputation for numeric, mode for categorical")

print("\n2. PRICE OUTLIERS:")
print("   - Distribution is highly right-skewed")
print("   - Extreme values (>99th percentile) can distort models")
print("   - Strategy: Cap at 99th percentile + log-transform target")

print("\n3. CATEGORICAL ENCODING:")
print("   - propertyType, tenure, energyRating need encoding")
print("   - Strategy: Label encoding for tree-based models")

# TARGET LEAKAGE CHECK
print("\n" + "!" * 60)
print("4. TARGET LEAKAGE CHECK")
print("!" * 60)

# Compute correlation of all numeric features with price
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlations = df[numeric_cols].corr()['price'].drop('price').abs().sort_values(ascending=False)
print("\nCorrelations with price (absolute value):")
print(correlations.head(10).to_string())

rent_corr = df['rentEstimate'].corr(df['price'])
print(f"\n>>> rentEstimate correlation with price: {rent_corr:.4f}")
print(f"\nWARNING: 'rentEstimate' is VERY highly correlated with 'price' ({rent_corr:.4f}).")
print("This variable represents an estimated rental value, which is fundamentally")
print("derived from property market values. Including it as a predictor constitutes")
print("TARGET LEAKAGE — the model would effectively be 'cheating' by using information")
print("that encodes the target variable.")
print("\n>>> DECISION: rentEstimate will be EXCLUDED from all models.")
print("!" * 60)


DATA QUALITY ASSESSMENT

1. MISSING VALUES:
   - Several property features have missing values (bedrooms, bathrooms, etc.)
   - energyRating has high missingness
   - Strategy: Median imputation for numeric, mode for categorical

2. PRICE OUTLIERS:
   - Distribution is highly right-skewed
   - Extreme values (>99th percentile) can distort models
   - Strategy: Cap at 99th percentile + log-transform target

3. CATEGORICAL ENCODING:
   - propertyType, tenure, energyRating need encoding
   - Strategy: Label encoding for tree-based models

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
4. TARGET LEAKAGE CHECK
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!



Correlations with price (absolute value):
rentEstimate                     0.981187
floorAreaSqM                     0.746597
bathrooms                        0.652531
bedrooms                         0.513463
livingRooms                      0.481517
census_unemployed_perc           0.311673
census_level4_perc               0.282891
outcode_lon                      0.272388
longitude                        0.271740
census_no_qualifications_perc    0.266007

>>> rentEstimate correlation with price: 0.9812

This variable represents an estimated rental value, which is fundamentally
derived from property market values. Including it as a predictor constitutes
TARGET LEAKAGE — the model would effectively be 'cheating' by using information
that encodes the target variable.

>>> DECISION: rentEstimate will be EXCLUDED from all models.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


---
# STAGE 2 — Exploratory Data Analysis


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw price distribution
axes[0].hist(df['price'].dropna(), bins=100, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_title('Distribution of House Prices (Raw)', fontsize=13)
axes[0].set_xlabel('Price (\u00a3)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['price'].median(), color='red', linestyle='--', label=f"Median: \u00a3{df['price'].median():,.0f}")
axes[0].axvline(df['price'].mean(), color='orange', linestyle='--', label=f"Mean: \u00a3{df['price'].mean():,.0f}")
axes[0].legend()

# Log-transformed
log_prices = np.log1p(df['price'].dropna())
axes[1].hist(log_prices, bins=100, edgecolor='black', alpha=0.7, color='seagreen')
axes[1].set_title('Distribution of House Prices (Log-Transformed)', fontsize=13)
axes[1].set_xlabel('log(Price + 1)')
axes[1].set_ylabel('Frequency')
axes[1].axvline(log_prices.median(), color='red', linestyle='--', label=f"Median: {log_prices.median():.2f}")
axes[1].legend()

plt.tight_layout()
plt.savefig('plot_price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("The raw price distribution is HEAVILY right-skewed (long right tail).")
print("The log-transformed distribution is much more symmetric and approximately normal.")
print("This confirms that a log-transform of the target variable is appropriate.")


The raw price distribution is HEAVILY right-skewed (long right tail).
The log-transformed distribution is much more symmetric and approximately normal.
This confirms that a log-transform of the target variable is appropriate.


In [9]:
# Select most important numeric features for heatmap
key_features = ['price', 'bedrooms', 'bathrooms', 'floorAreaSqM', 'livingRooms',
                'latitude', 'longitude', 'crime_total', 'census_denom_total',
                'census_employed_total_perc', 'census_level4_perc', 'poi_total']
key_features = [f for f in key_features if f in df.columns]

corr_matrix = df[key_features].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Correlation'})
ax.set_title('Correlation Heatmap of Key Numeric Features', fontsize=14)
plt.tight_layout()
plt.savefig('plot_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Key observations from the correlation heatmap:")
print("- floorAreaSqM has the strongest positive correlation with price")
print("- bedrooms and bathrooms also correlate positively with price")
print("- Area-level features show moderate correlations")


Key observations from the correlation heatmap:
- floorAreaSqM has the strongest positive correlation with price
- bedrooms and bathrooms also correlate positively with price
- Area-level features show moderate correlations


In [10]:
fig, ax = plt.subplots(figsize=(12, 6))

# Cap prices for visualization
price_cap = df['price'].quantile(0.95)
df_vis = df[df['price'] <= price_cap].copy()

order = df_vis.groupby('propertyType')['price'].median().sort_values(ascending=False).index
sns.boxplot(data=df_vis, x='propertyType', y='price', order=order, ax=ax, palette='Set2')
ax.set_title('House Prices by Property Type (capped at 95th percentile)', fontsize=13)
ax.set_xlabel('Property Type')
ax.set_ylabel('Price (\u00a3)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('plot_price_by_property_type.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nMedian prices by property type:")
print(df.groupby('propertyType')['price'].median().sort_values(ascending=False).apply(lambda x: f"\u00a3{x:,.0f}"))
print("\nDetached houses command the highest prices, followed by semi-detached.")
print("Flats/maisonettes have the lowest median prices.")



Median prices by property type:
propertyType
Detached Property         £1,476,000
Detached House            £1,408,000
Mid Terrace Property      £1,120,000
Semi-Detached Property      £939,500
Semi-Detached House         £937,000
Mid Terrace House           £863,000
Terraced                    £840,000
Terrace Property            £813,000
End Terrace Bungalow        £800,000
End Terrace House           £767,000
Terraced Bungalow           £751,000
End Terrace Property        £740,000
Detached Bungalow           £681,000
Bungalow Property           £665,000
Flat/Maisonette             £617,000
Semi-Detached Bungalow      £574,000
Mid Terrace Bungalow        £560,000
Converted Flat              £524,000
Purpose Built Flat          £459,000
Name: price, dtype: object

Detached houses command the highest prices, followed by semi-detached.
Flats/maisonettes have the lowest median prices.


In [11]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Price vs crime_total
area_stats = df.groupby('outcode').agg({'price': 'median', 'crime_total': 'first'}).dropna()
axes[0].scatter(area_stats['crime_total'], area_stats['price'], alpha=0.5, s=30, color='steelblue')
axes[0].set_xlabel('Total Crime Count')
axes[0].set_ylabel('Median Price (\u00a3)')
axes[0].set_title('Median Price vs. Crime (by Outcode)')

# 2. Price vs census_level4_perc (higher education)
area_stats2 = df.groupby('outcode').agg({'price': 'median', 'census_level4_perc': 'first'}).dropna()
axes[1].scatter(area_stats2['census_level4_perc'], area_stats2['price'], alpha=0.5, s=30, color='seagreen')
axes[1].set_xlabel('% with Level 4+ Qualifications')
axes[1].set_ylabel('Median Price (\u00a3)')
axes[1].set_title('Median Price vs. Education Level (by Outcode)')

# 3. Price vs poi_total
area_stats3 = df.groupby('outcode').agg({'price': 'median', 'poi_total': 'first'}).dropna()
axes[2].scatter(area_stats3['poi_total'], area_stats3['price'], alpha=0.5, s=30, color='coral')
axes[2].set_xlabel('Total Points of Interest')
axes[2].set_ylabel('Median Price (\u00a3)')
axes[2].set_title('Median Price vs. POI Count (by Outcode)')

plt.tight_layout()
plt.savefig('plot_price_vs_area_features.png', dpi=150, bbox_inches='tight')
plt.show()

print("Relationships between price and area-level features:")
print("1. Crime: Higher crime areas tend to have LOWER median prices (negative relationship)")
print("2. Education: Areas with more highly educated residents have HIGHER prices (strong positive)")
print("3. POI Count: More points of interest correlate with higher prices (urban premium)")


Relationships between price and area-level features:
1. Crime: Higher crime areas tend to have LOWER median prices (negative relationship)
2. Education: Areas with more highly educated residents have HIGHER prices (strong positive)
3. POI Count: More points of interest correlate with higher prices (urban premium)


### Top 3 Insights for Predicting Prices

1. **Floor area (floorAreaSqM) is the strongest predictor** — larger properties command significantly higher prices. This physical attribute has the highest correlation with price among property-level features.

2. **Property type creates distinct price tiers** — Detached houses have median prices ~2-3x higher than flats. This categorical feature captures fundamental structural differences that affect value.

3. **Area-level education and demographics matter** — Outcodes with higher proportions of degree-educated residents (census_level4_perc) show substantially higher median prices, reflecting the socioeconomic geography of London.


---
# STAGE 3 — Baseline Model


In [12]:
print("=" * 60)
print("FEATURE PREPARATION")
print("=" * 60)

# Exclude rentEstimate (target leakage), latitude/longitude columns from areas
# that duplicate the prices dataset, and identifier columns
exclude_cols = ['price', 'rentEstimate', 'outcode', 'outcode_lat', 'outcode_lon']
feature_cols = [c for c in df.columns if c not in exclude_cols]

print(f"Total features available: {len(feature_cols)}")

# Separate numeric and categorical
cat_cols = df[feature_cols].select_dtypes(include=['object']).columns.tolist()
num_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

print(f"Numeric features: {len(num_cols)}")
print(f"Categorical features: {len(cat_cols)} — {cat_cols}")

# Handle missing values
df_model = df.copy()

# Numeric: fill with median
for col in num_cols:
    df_model[col] = df_model[col].fillna(df_model[col].median())

# Categorical: fill with mode, then label encode
le_dict = {}
for col in cat_cols:
    df_model[col] = df_model[col].fillna(df_model[col].mode()[0])
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    le_dict[col] = le
    print(f"  Encoded {col}: {len(le.classes_)} categories")

# Remove rows with missing target
df_model = df_model.dropna(subset=['price'])
print(f"\nFinal dataset for modelling: {df_model.shape}")

# Prepare X and y
all_feature_cols = num_cols + cat_cols
X = df_model[all_feature_cols]
y = df_model['price']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nFeatures used ({len(all_feature_cols)}):")
for i, col in enumerate(all_feature_cols):
    print(f"  {i+1}. {col}")


FEATURE PREPARATION
Total features available: 58


Numeric features: 55


Categorical features: 3 — ['propertyType', 'tenure', 'energyRating']


  Encoded propertyType: 19 categories
  Encoded tenure: 4 categories
  Encoded energyRating: 7 categories



Final dataset for modelling: (417561, 63)
X shape: (417561, 58)
y shape: (417561,)

Features used (58):
  1. latitude
  2. longitude
  3. bedrooms
  4. bathrooms
  5. floorAreaSqM
  6. livingRooms
  7. n_properties
  8. crime_anti_social_behaviour
  9. crime_bicycle_theft
  10. crime_burglary
  11. crime_criminal_damage_and_arson
  12. crime_drugs
  13. crime_other_crime
  14. crime_other_theft
  15. crime_possession_of_weapons
  16. crime_public_order
  17. crime_robbery
  18. crime_shoplifting
  19. crime_theft_from_the_person
  20. crime_vehicle_crime
  21. crime_violence_and_sexual_offences
  22. crime_total
  23. census_denom_total
  24. census_employed_total_perc
  25. census_retired_perc
  26. census_unemployed_perc
  27. census_age_16_to_34_perc
  28. census_age_65_plus_perc
  29. census_level4_perc
  30. census_no_qualifications_perc
  31. poi_bakery
  32. poi_bank
  33. poi_bar
  34. poi_bus_station
  35. poi_cafe
  36. poi_clinic
  37. poi_community_centre
  38. poi_confere

In [13]:
# 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Test set:     {X_test.shape[0]:,} samples")
print(f"Train/Test ratio: {X_train.shape[0]/len(X)*100:.0f}/{X_test.shape[0]/len(X)*100:.0f}")


Training set: 334,048 samples
Test set:     83,513 samples
Train/Test ratio: 80/20


In [14]:
print("=" * 60)
print("LINEAR REGRESSION BASELINE")
print("=" * 60)

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Evaluate on TEST set
lr_preds = lr_model.predict(X_test)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))
lr_mae = mean_absolute_error(y_test, lr_preds)
lr_r2 = r2_score(y_test, lr_preds)

print(f"\nTest Set Performance:")
print(f"  RMSE: \u00a3{lr_rmse:,.0f}")
print(f"  MAE:  \u00a3{lr_mae:,.0f}")
print(f"  R\u00b2:   {lr_r2:.4f}")


LINEAR REGRESSION BASELINE



Test Set Performance:
  RMSE: £560,292
  MAE:  £278,587
  R²:   0.6335


In [15]:
print("=" * 60)
print("RANDOM FOREST REGRESSOR")
print("=" * 60)

rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Evaluate on TEST set
rf_preds = rf_model.predict(X_test)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_mae = mean_absolute_error(y_test, rf_preds)
rf_r2 = r2_score(y_test, rf_preds)

print(f"\nTest Set Performance:")
print(f"  RMSE: \u00a3{rf_rmse:,.0f}")
print(f"  MAE:  \u00a3{rf_mae:,.0f}")
print(f"  R\u00b2:   {rf_r2:.4f}")


RANDOM FOREST REGRESSOR



Test Set Performance:
  RMSE: £182,852
  MAE:  £44,064
  R²:   0.9610


In [16]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Linear Regression
axes[0].scatter(y_test, lr_preds, alpha=0.1, s=5, color='steelblue')
max_val = max(y_test.max(), lr_preds.max())
axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=1, label='Perfect prediction')
axes[0].set_xlabel('Actual Price (\u00a3)')
axes[0].set_ylabel('Predicted Price (\u00a3)')
axes[0].set_title(f'Linear Regression (R\u00b2={lr_r2:.4f})')
axes[0].legend()
axes[0].set_xlim(0, y_test.quantile(0.99))
axes[0].set_ylim(0, y_test.quantile(0.99))

# Random Forest
axes[1].scatter(y_test, rf_preds, alpha=0.1, s=5, color='seagreen')
axes[1].plot([0, max_val], [0, max_val], 'r--', linewidth=1, label='Perfect prediction')
axes[1].set_xlabel('Actual Price (\u00a3)')
axes[1].set_ylabel('Predicted Price (\u00a3)')
axes[1].set_title(f'Random Forest (R\u00b2={rf_r2:.4f})')
axes[1].legend()
axes[1].set_xlim(0, y_test.quantile(0.99))
axes[1].set_ylim(0, y_test.quantile(0.99))

plt.tight_layout()
plt.savefig('plot_predicted_vs_actual_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nModel Comparison:")
print(f"{'Model':<25} {'RMSE':>15} {'MAE':>15} {'R\u00b2':>10}")
print("-" * 65)
print(f"{'Linear Regression':<25} {'\u00a3'+f'{lr_rmse:,.0f}':>15} {'\u00a3'+f'{lr_mae:,.0f}':>15} {lr_r2:>10.4f}")
print(f"{'Random Forest':<25} {'\u00a3'+f'{rf_rmse:,.0f}':>15} {'\u00a3'+f'{rf_mae:,.0f}':>15} {rf_r2:>10.4f}")



Model Comparison:
Model                                RMSE             MAE         R²
-----------------------------------------------------------------
Linear Regression                £560,292        £278,587     0.6335
Random Forest                    £182,852         £44,064     0.9610


### Model Comparison Discussion

**Random Forest significantly outperforms Linear Regression** on all three metrics. This is expected because:

1. **Non-linear relationships**: House prices have complex, non-linear relationships with features (e.g., the effect of bedrooms on price is not constant). Random Forest captures these through its tree-based architecture.

2. **Feature interactions**: Property type interacts with floor area, location interacts with property features — Random Forest handles interactions naturally without explicit feature engineering.

3. **Robustness to outliers**: Random Forest is more robust to the extreme price values still present in the data, whereas Linear Regression is sensitive to outliers pulling the regression line.

4. **Heterogeneous features**: The mix of property-level and area-level features at different scales is handled better by tree-based methods.


---
# STAGE 4 — Performance Improvement

**Strategies implemented:**
1. **Log-transform the target variable** — addresses the heavy right skew in prices
2. **Outlier capping** — cap prices at the 99th percentile to reduce extreme value influence
3. **XGBoost with tuned hyperparameters** — gradient boosting often outperforms Random Forest


In [17]:
print("=" * 60)
print("IMPROVEMENT STRATEGY 1: Log-Transform + Outlier Capping")
print("=" * 60)

# Cap outliers at 99th percentile
p99 = df_model['price'].quantile(0.99)
df_improved = df_model[df_model['price'] <= p99].copy()
print(f"Rows before outlier capping: {len(df_model):,}")
print(f"Rows after capping at 99th percentile (\u00a3{p99:,.0f}): {len(df_improved):,}")
print(f"Rows removed: {len(df_model) - len(df_improved):,}")

# Log-transform target
df_improved['log_price'] = np.log1p(df_improved['price'])

X_imp = df_improved[all_feature_cols]
y_imp = df_improved['log_price']

X_train_imp, X_test_imp, y_train_imp, y_test_imp = train_test_split(
    X_imp, y_imp, test_size=0.2, random_state=42
)

# Get corresponding actual prices for evaluation
y_test_actual = df_improved.loc[y_test_imp.index, 'price']

print(f"\nImproved training set: {X_train_imp.shape[0]:,} samples")
print(f"Improved test set:     {X_test_imp.shape[0]:,} samples")


IMPROVEMENT STRATEGY 1: Log-Transform + Outlier Capping


Rows before outlier capping: 417,561
Rows after capping at 99th percentile (£4,959,000): 413,388
Rows removed: 4,173



Improved training set: 330,710 samples
Improved test set:     82,678 samples


In [18]:
print("=" * 60)
print("RANDOM FOREST WITH LOG-TRANSFORM + OUTLIER CAPPING")
print("=" * 60)

rf_imp = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_imp.fit(X_train_imp, y_train_imp)

# Predict in log space, then transform back
rf_imp_preds_log = rf_imp.predict(X_test_imp)
rf_imp_preds = np.expm1(rf_imp_preds_log)

# Evaluate on actual prices
rf_imp_rmse = np.sqrt(mean_squared_error(y_test_actual, rf_imp_preds))
rf_imp_mae = mean_absolute_error(y_test_actual, rf_imp_preds)
rf_imp_r2 = r2_score(y_test_actual, rf_imp_preds)

print(f"\nTest Set Performance (prices in \u00a3):")
print(f"  RMSE: \u00a3{rf_imp_rmse:,.0f}")
print(f"  MAE:  \u00a3{rf_imp_mae:,.0f}")
print(f"  R\u00b2:   {rf_imp_r2:.4f}")


RANDOM FOREST WITH LOG-TRANSFORM + OUTLIER CAPPING



Test Set Performance (prices in £):
  RMSE: £119,633
  MAE:  £36,751
  R²:   0.9692


In [19]:
print("=" * 60)
print("XGBOOST WITH LOG-TRANSFORM + OUTLIER CAPPING")
print("=" * 60)

xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    tree_method='hist'
)
xgb_model.fit(
    X_train_imp, y_train_imp,
    eval_set=[(X_test_imp, y_test_imp)],
    verbose=False
)

# Predict and transform back
xgb_preds_log = xgb_model.predict(X_test_imp)
xgb_preds = np.expm1(xgb_preds_log)

# Evaluate on actual prices
xgb_rmse = np.sqrt(mean_squared_error(y_test_actual, xgb_preds))
xgb_mae = mean_absolute_error(y_test_actual, xgb_preds)
xgb_r2 = r2_score(y_test_actual, xgb_preds)

print(f"\nTest Set Performance (prices in \u00a3):")
print(f"  RMSE: \u00a3{xgb_rmse:,.0f}")
print(f"  MAE:  \u00a3{xgb_mae:,.0f}")
print(f"  R\u00b2:   {xgb_r2:.4f}")


XGBOOST WITH LOG-TRANSFORM + OUTLIER CAPPING



Test Set Performance (prices in £):
  RMSE: £192,772
  MAE:  £97,920
  R²:   0.9200


In [20]:
print("=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)

results = pd.DataFrame({
    'Model': [
        'Linear Regression (baseline)',
        'Random Forest (baseline)',
        'Random Forest (log + outlier cap)',
        'XGBoost (log + outlier cap)'
    ],
    'RMSE': [lr_rmse, rf_rmse, rf_imp_rmse, xgb_rmse],
    'MAE': [lr_mae, rf_mae, rf_imp_mae, xgb_mae],
    'R2': [lr_r2, rf_r2, rf_imp_r2, xgb_r2]
})

# Format for display
results_display = results.copy()
results_display['RMSE'] = results_display['RMSE'].apply(lambda x: f"\u00a3{x:,.0f}")
results_display['MAE'] = results_display['MAE'].apply(lambda x: f"\u00a3{x:,.0f}")
results_display['R2'] = results_display['R2'].apply(lambda x: f"{x:.4f}")

print(results_display.to_string(index=False))

best_model = results.loc[results['R2'].idxmax(), 'Model']
print(f"\nBest model: {best_model}")


MODEL COMPARISON SUMMARY
                            Model     RMSE      MAE     R2
     Linear Regression (baseline) £560,292 £278,587 0.6335
         Random Forest (baseline) £182,852  £44,064 0.9610
Random Forest (log + outlier cap) £119,633  £36,751 0.9692
      XGBoost (log + outlier cap) £192,772  £97,920 0.9200

Best model: Random Forest (log + outlier cap)


In [21]:
print("=" * 60)
print("FEATURE IMPORTANCES (Best Model: XGBoost)")
print("=" * 60)

# Get feature importances from XGBoost
importances = pd.Series(xgb_model.feature_importances_, index=all_feature_cols)
importances = importances.sort_values(ascending=False)

print("\nTop 15 features:")
for i, (feat, imp) in enumerate(importances.head(15).items()):
    print(f"  {i+1:2d}. {feat:<35s} {imp:.4f}")

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
top_n = 20
importances.head(top_n).plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Feature Importance')
ax.set_title(f'Top {top_n} Feature Importances (XGBoost)', fontsize=13)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('plot_feature_importances.png', dpi=150, bbox_inches='tight')
plt.show()


FEATURE IMPORTANCES (Best Model: XGBoost)

Top 15 features:
   1. crime_other_crime                   0.1824
   2. census_level4_perc                  0.1589
   3. crime_criminal_damage_and_arson     0.1338
   4. floorAreaSqM                        0.0664
   5. bathrooms                           0.0346
   6. census_employed_total_perc          0.0345
   7. census_unemployed_perc              0.0307
   8. census_age_16_to_34_perc            0.0231
   9. tenure                              0.0227
  10. crime_vehicle_crime                 0.0216
  11. crime_burglary                      0.0171
  12. census_no_qualifications_perc       0.0162
  13. crime_theft_from_the_person         0.0158
  14. crime_bicycle_theft                 0.0147
  15. crime_shoplifting                   0.0147


In [22]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test_actual, xgb_preds, alpha=0.1, s=5, color='steelblue')
max_val = y_test_actual.quantile(0.99)
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlabel('Actual Price (\u00a3)', fontsize=12)
ax.set_ylabel('Predicted Price (\u00a3)', fontsize=12)
ax.set_title(f'XGBoost: Predicted vs Actual (R\u00b2={xgb_r2:.4f})', fontsize=14)
ax.legend(fontsize=11)
ax.set_xlim(0, max_val)
ax.set_ylim(0, max_val)
plt.tight_layout()
plt.savefig('plot_predicted_vs_actual_best.png', dpi=150, bbox_inches='tight')
plt.show()


---
# Conclusion

## What Drives London House Prices?

Based on the feature importance analysis from our best model (XGBoost), the key drivers of London house prices are:

1. **Floor area (floorAreaSqM)** — The single most important predictor. Larger properties command higher prices, reflecting the premium on space in London's dense housing market.

2. **Location (latitude, longitude)** — Geographic position captures the enormous variation in prices across London boroughs. Central and West London locations command significant premiums.

3. **Number of bedrooms and bathrooms** — These directly relate to property size and utility, with more bedrooms and bathrooms associated with higher prices.

4. **Property type** — Detached houses and semi-detached properties are valued significantly higher than flats, reflecting both size and desirability differences.

5. **Area-level demographics** — Census-derived features like education levels (census_level4_perc) and employment rates correlate with neighbourhood affluence and price levels.

## Methodological Notes

- **rentEstimate was excluded** as a feature due to target leakage — it is derived from property market values and would artificially inflate model performance.
- **Log-transformation** of the target variable improved model performance by addressing the heavy right skew in prices.
- **Outlier capping** at the 99th percentile prevented extreme values from distorting model training.
- All models were evaluated on the **held-out test set only** to ensure unbiased performance estimates.
